# Normalize Morita et al. lipid names to GOSLIN format

**Goal:** Convert proprietary lipid names from Morita et al. into GOSLIN-normalized form and validate them against the GOSLIN REST API.

**Steps:**
1. Load raw lipidomics data (`lipid_Morita_et_al.xlsx`)
2. Normalize lipid names to GOSLIN-compatible format (regex-based pre-processing)
3. Validate normalized names via GOSLIN REST API and retrieve hierarchy metadata
4. Calculate and report match rate
5. Save results to `data/processed/goslin_Morita.csv`

In [1]:
import re
import time
import requests
import pandas as pd
from urllib.parse import quote

## Step 1 — Load raw lipidomics data

In [2]:
# Load unique lipid names from the main sheet of Morita et al.
df_raw = pd.read_excel('../data/raw/lipid_Morita_et_al.xlsx', sheet_name='Lipid_all')

lipid_names = df_raw['CompoundName'].dropna().unique().tolist()

print(f"Total unique lipid names: {len(lipid_names)}")
lipid_names[:5]

Total unique lipid names: 1306


['Cer d18:1_14:0',
 'Cer d18:1_16:0',
 'Cer d18:1_16:1',
 'Cer d18:1_18:0',
 'Cer d18:1_18:1']

## Step 2 — Normalize lipid names to GOSLIN-compatible format

Morita et al. uses a proprietary naming convention that differs from GOSLIN in three ways:

| Pattern | Morita format | GOSLIN format |
|---|---|---|
| Multi-chain with parentheses | `TG(12:0)(14:1)(18:2)` | `TG 12:0/14:1/18:2` |
| Ceramide with `d` prefix | `Cer d18:1_16:0` | `Cer 18:1;2/16:0` |
| Underscore chain separator | `PC 16:0_18:1` | `PC 16:0/18:1` |

In [4]:
def normalize_lipid_name(name):
    """
    Convert Morita et al. proprietary lipid names to GOSLIN-compatible format.

    Rules applied in order:
    1. Parenthesis chains -> underscore-separated (molecular species: sn-position unknown)
       e.g. TG(12:0)(14:1)(18:2) -> TG 12:0_14:1_18:2
    2. Ceramide 'd' prefix -> ';2' hydroxyl notation
       e.g. Cer d18:1_16:0 -> Cer 18:1;2/16:0

    NOTE: '_' and '/' are NOT interchangeable.
    '_' = molecular species (sn-position unknown, LIPID MAPS Shorthand)
    '/' = structural subspecies (sn-position known)
    Morita et al. use '_' throughout, so Rule 3 (underscore -> slash) has been removed.
    """
    # Rule 1: parenthesis-enclosed chains (handles 1-, 2-, 3-chain lipids)
    # Use '_' (not '/') because Morita's parenthesis notation does not specify sn-positions.
    m = re.match(r'^([A-Za-z][A-Za-z0-9]*)((?:\([^)]+\))+)$', name)
    if m:
        cls    = m.group(1)
        chains = re.findall(r'\(([^)]+)\)', m.group(2))
        name   = cls + ' ' + '_'.join(chains)

    # Rule 2: Ceramide 'd' prefix (dihydroxy sphingoid base)
    # The '/' here separates the sphingoid base from the acyl chain (GOSLIN convention).
    name = re.sub(r'((?:Hex)?Cer) d(\d+:\d+)_(\S+)', r'\1 \2;2/\3', name)

    return name


# Apply normalization to all lipid names
lipid_names_normalized = [normalize_lipid_name(n) for n in lipid_names]

# Spot-check conversion examples
examples = [
    'TG(12:0)(14:1)(18:2)',
    'PC(16:0)(18:1)',
    'MG(18:1)',
    'Cer d18:1_14:0',
    'HexCer d18:1_24:1',
    'PC 16:0_18:1',
    'Cholesterol',
]
print("Normalization examples:")
for n in examples:
    print(f"  {n:35s} -> {normalize_lipid_name(n)}")

Normalization examples:
  TG(12:0)(14:1)(18:2)                -> TG 12:0_14:1_18:2
  PC(16:0)(18:1)                      -> PC 16:0_18:1
  MG(18:1)                            -> MG 18:1
  Cer d18:1_14:0                      -> Cer 18:1;2/14:0
  HexCer d18:1_24:1                   -> HexCer 18:1;2/24:1
  PC 16:0_18:1                        -> PC 16:0_18:1
  Cholesterol                         -> Cholesterol


## Step 3 — Validate via GOSLIN REST API and retrieve hierarchy metadata

Send each normalized name to the GOSLIN REST API. On success, retrieve:
- `normalized_name`: GOSLIN canonical form
- `lipid_level`: granularity level (SPECIES, MOLECULAR_SPECIES, SN_POSITION, …)
- `category` / `class` / `class_name`: LipidMaps classification
- `mass`, `formula`: physicochemical properties

Failed names (class-only names without chain info, e.g. `"PC"`, `"TG"`) are collected separately.

> **Note:** The API has a rate limit; `time.sleep(0.1)` is applied between requests.

In [5]:
import os

CACHE_PATH = '../data/processed/goslin_Morita.csv'
GOSLIN_API = "https://metabocloud.mesocentre.uca.fr/goslin/validate"

if os.path.exists(CACHE_PATH):
    # ── Cache hit: load from CSV, skip API calls ──────────────────────────────
    print(f"Cache found: loading from '{CACHE_PATH}' (delete to re-fetch)")
    df_goslin = pd.read_csv(CACHE_PATH)

    # Reconstruct results / failed from the cached DataFrame for Step 4
    results = {
        row['original_name']: row.drop('original_name').to_dict()
        for _, row in df_goslin.iterrows()
    }
    matched_originals = set(df_goslin['original_name'])
    failed = [n for n in lipid_names if n not in matched_originals]

else:
    # ── Cache miss: call GOSLIN REST API (runs once, then cached) ─────────────
    print("No cache found — calling GOSLIN REST API (this may take a few minutes)...")

    results = {}
    failed  = []

    for original, normalized in zip(lipid_names, lipid_names_normalized):
        try:
            response = requests.get(f"{GOSLIN_API}?lipid_names={quote(normalized)}")
            data     = response.json()

            if data['nb_success'] > 0:
                lipid = data['lipid_list'][0]
                results[original] = {
                    'normalized_name': lipid.get('normalized_name'),
                    'lipid_level':     lipid.get('lipid_level'),
                    'category':        lipid.get('lipidmaps_category'),
                    'class':           lipid.get('lipidmaps_class'),
                    'class_name':      lipid.get('class_name'),
                    'extended_class':  lipid.get('extended_class'),
                    'mass':            lipid.get('mass'),
                    'formula':         lipid.get('formula'),
                }
            else:
                failed.append(original)

            time.sleep(0.1)

        except Exception as e:
            print(f"Error for '{original}': {e}")
            failed.append(original)

    print(f"Done — API returned {len(results)} successes, {len(failed)} failures")

Cache found: loading from '../data/processed/goslin_Morita.csv' (delete to re-fetch)


## Step 4 — Match rate and results

In [6]:
n_total   = len(lipid_names)
n_matched = len(results)
n_failed  = len(failed)

print(f"Total lipid names  : {n_total}")
print(f"Matched (success)  : {n_matched}  ({n_matched / n_total * 100:.1f}%)")
print(f"Unmatched (failed) : {n_failed}   ({n_failed  / n_total * 100:.1f}%)")
print()
print("Unmatched names (typically class-level names without chain info):")
for name in failed:
    print(f"  {name}")

# Build result DataFrame
df_goslin = (
    pd.DataFrame.from_dict(results, orient='index')
    .rename_axis('original_name')
    .reset_index()
)[['original_name', 'normalized_name', 'lipid_level',
   'category', 'class', 'class_name', 'extended_class', 'mass', 'formula']]

print(f"\nResult shape: {df_goslin.shape}")
df_goslin.head()

Total lipid names  : 1306
Matched (success)  : 1290  (98.8%)
Unmatched (failed) : 16   (1.2%)

Unmatched names (typically class-level names without chain info):
  Cer
  DG
  LPC
  LPE
  MG
  PA
  PC
  PE
  PG
  PI
  PS
  HexCer
  SM
  CE
  FA
  TG

Result shape: (1290, 9)


,original_name,normalized_name,lipid_level,category,class,class_name,extended_class,mass,formula
0,Cer d18:1_14:0,NaN,SN_POSITION,SP,Ceramides [SP02],Cer,Cer,509.480795,C32H63NO3
1,Cer d18:1_16:0,NaN,SN_POSITION,SP,Ceramides [SP02],Cer,Cer,537.512095,C34H67NO3
2,Cer d18:1_16:1,NaN,SN_POSITION,SP,Ceramides [SP02],Cer,Cer,535.496445,C34H65NO3
3,Cer d18:1_18:0,NaN,SN_POSITION,SP,Ceramides [SP02],Cer,Cer,565.543395,C36H71NO3
4,Cer d18:1_18:1,NaN,SN_POSITION,SP,Ceramides [SP02],Cer,Cer,563.527745,C36H69NO3


In [8]:
# GOSLIN lipid_level distribution
# Finest to coarsest: FULL_STRUCTURE > COMPLETE_STRUCTURE > SN_POSITION >
#                     STRUCTURE_DEFINED > MOLECULAR_SPECIES > SPECIES > CLASS
GOSLIN_LEVEL_ORDER = [
    'FULL_STRUCTURE',
    'COMPLETE_STRUCTURE',
    'SN_POSITION',
    'STRUCTURE_DEFINED',
    'MOLECULAR_SPECIES',
    'SPECIES',
    'CLASS',
    'UNDEFINED',
]

level_counts = df_goslin['lipid_level'].value_counts(dropna=False)
n_total = len(df_goslin)

rows = []
for lv in GOSLIN_LEVEL_ORDER:
    n = int(level_counts.get(lv, 0))
    if n > 0:
        rows.append({'lipid_level': lv, 'n': n, 'pct': f'{n / n_total * 100:.1f}%'})

n_other = int(df_goslin['lipid_level'].isna().sum())
if n_other > 0:
    rows.append({'lipid_level': '(unmatched)', 'n': n_other, 'pct': f'{n_other / n_total * 100:.1f}%'})

df_level_summary = pd.DataFrame(rows)
print(f'GOSLIN lipid_level summary (n={n_total}):')
print()
print(df_level_summary.to_string(index=False))

GOSLIN lipid_level summary (n=1290):

       lipid_level    n   pct
    FULL_STRUCTURE    1  0.1%
COMPLETE_STRUCTURE   36  2.8%
       SN_POSITION 1172 90.9%
 STRUCTURE_DEFINED   12  0.9%
 MOLECULAR_SPECIES   43  3.3%
           SPECIES   26  2.0%


In [9]:
# Save to CSV (avoids re-running the API calls in downstream notebooks)
out_path = '../data/processed/goslin_Morita.csv'
df_goslin.to_csv(out_path, index=False)
print(f"Saved: {out_path}")

Saved: ../data/processed/goslin_Morita.csv
